# Time & cost pairwise — Wilcoxon signed-rank

Pairwise wall-clock comparisons with Bonferroni correction and effect sizes.

In [ ]:
from analysis import load_runs

# Pin matplotlib (Agg) + RNG seeds for reproducible, headless figures.
plt = load_runs.pin_style(seed=0)

# Load real runs if any exist, else the deterministic synthetic dataset.
records = load_runs.load_all()
backends = load_runs.backend_names(records)
STRATEGY = "zero_shot"  # the held-fixed strategy for the cross-backend tests
print(f"{len(records)} runs, backends={backends}")

In [ ]:
from analysis import stats as st
from analysis.aggregate import aggregate_runs, build_metric_matrix

summaries = aggregate_runs(records)
metric = "wall_clock"
ids, mat = build_metric_matrix(summaries, metric=metric, backends=backends, strategy=STRATEGY)
cols = {b: [row[j] for row in mat] for j, b in enumerate(backends)}
pairs = st.pairwise_labels(backends)
alpha = st.bonferroni_alpha(0.05, len(pairs))
results = [st.wilcoxon_signed_rank(cols[a], cols[b], label_a=a, label_b=b) for a, b in pairs]
for r in results:
    sig = "*" if r.p_value < alpha else " "
    print(
        f"{sig} {r.backend_a} vs {r.backend_b}: p={r.p_value:.4g} "
        f"r={r.rank_biserial_r:+.3f} ({r.method})"
    )
print(f"Bonferroni-corrected alpha = {alpha:.4f}")

In [ ]:
labels = [f"{r.backend_a}\nvs\n{r.backend_b}" for r in results]
fig, ax = plt.subplots()
colors = ["seagreen" if r.p_value < alpha else "gray" for r in results]
ax.bar(labels, [r.rank_biserial_r for r in results], color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("rank-biserial r")
ax.set_title(f"Wilcoxon pairwise effect sizes ({metric})")
load_runs.save_figure(fig, "04_time_cost_pairwise", "wilcoxon_effect_sizes")
fig

In [ ]:
from IPython.display import Markdown

lines = [
    f"| {r.backend_a} | {r.backend_b} | {r.p_value:.4g} | {r.rank_biserial_r:+.3f} | "
    f"{r.method} | {'yes' if r.p_value < alpha else 'no'} |"
    for r in results
]
Markdown(
    f"### Summary — time/cost pairwise (Wilcoxon, Bonferroni)\n\n"
    f"Metric **{metric}**; corrected alpha = **{alpha:.4f}**.\n\n"
    "| A | B | p | rank-biserial r | method | sig |\n|---|---|---|---|---|---|\n" + "\n".join(lines)
)